# Module 04 — ML Theory & Evaluation
## 04-04: Data Augmentation & Color Spaces

**Objective:** Build a complete image augmentation toolkit from scratch — geometric transforms, color-space conversions, and advanced mixing strategies (Mixup, CutMix) — then apply them to CIFAR-10 to measure their impact on generalization.

**Prerequisites:** 1-04 (NumPy broadcasting), 1-05 (Matplotlib & visualization)

## Part 0 — Setup & Prerequisites

Data augmentation is a form of **regularization** that injects domain-informed invariances into training.
Rather than adding a penalty term to the loss, augmentation expands the effective dataset by showing the
model the same image under different geometric and photometric transformations — encouraging it to learn
features that are stable under those changes.  This notebook implements the full augmentation pipeline
from scratch using NumPy, then validates against torchvision's battle-tested transforms.

> **Note:** The focus is the augmentation methodology. We train a lightweight CNN for only a few epochs
> to measure the effect of augmentation on generalization, not to achieve state-of-the-art accuracy.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import random
import warnings
from typing import Callable
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from torch.utils.data import DataLoader
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = "../data"
BATCH_SIZE = 128
NUM_EPOCHS = 5          # kept short — focus is augmentation methodology
LEARNING_RATE = 1e-3
NUM_CLASSES = 10
IMG_SIZE = 32           # CIFAR-10 native resolution

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

### Data Loading & Exploration

We load CIFAR-10 without augmentation first so we can visualise the raw images and understand the
baseline before applying any transforms.

In [ ]:
# ── Load CIFAR-10 (no augmentation — raw pixel values for visualisation) ─────
bare_transform = T.Compose([
    T.ToTensor(),   # HWC uint8 → CHW float32 in [0, 1]
])

full_train_raw = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=bare_transform
)
test_set_raw = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=bare_transform
)

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Train samples : {len(full_train_raw):,}")
print(f"Test  samples : {len(test_set_raw):,}")
print(f"Image shape   : {full_train_raw[0][0].shape}   (C, H, W)")
print(f"Classes       : {CIFAR10_CLASSES}")

# Class distribution (balanced by design)
labels_all = [full_train_raw[i][1] for i in range(len(full_train_raw))]
counts = np.bincount(labels_all)
print("\nClass counts:")
for cls_id, (name, cnt) in enumerate(zip(CIFAR10_CLASSES, counts)):
    print(f"  {cls_id}: {name:12s} {cnt:,}")

In [ ]:
# ── Visualise sample images ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
shown: dict = {c: False for c in range(NUM_CLASSES)}
idx = 0
col = 0
for img_tensor, label in full_train_raw:
    if not shown[label]:
        img_np = img_tensor.permute(1, 2, 0).numpy()  # CHW → HWC
        axes[0, label].imshow(img_np)
        axes[0, label].set_title(CIFAR10_CLASSES[label], fontsize=8)
        axes[0, label].axis("off")
        shown[label] = True
    if all(shown.values()):
        break

# Second row: same classes, second example
shown2: dict = {c: 0 for c in range(NUM_CLASSES)}
for img_tensor, label in full_train_raw:
    shown2[label] += 1
    if shown2[label] == 2:
        img_np = img_tensor.permute(1, 2, 0).numpy()
        axes[1, label].imshow(img_np)
        axes[1, label].axis("off")
    if all(v >= 2 for v in shown2.values()):
        break

plt.suptitle("CIFAR-10: Two examples per class (raw, no augmentation)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Part 1 — Augmentation Primitives from Scratch

We implement each augmentation as a callable that operates on NumPy arrays of shape **(H, W, C)**
in the range **[0, 1]** (float32).  After implementation we visualise and compare against torchvision.

### 1.1 Geometric Transforms

Geometric transforms alter **spatial structure** while preserving pixel intensities:

| Transform | Invariance taught |
|-----------|------------------|
| Horizontal flip | Left–right symmetry |
| Random crop | Translation, scale |
| Rotation | Rotational symmetry |
| Affine warp | Combined shear + rotation + scale |

In [ ]:
# ── Geometric Transforms ──────────────────────────────────────────────────────

def horizontal_flip(image: np.ndarray, p: float = 0.5) -> np.ndarray:
    '''Flip image left-to-right with probability p.

    Args:
        image: HWC float32 array in [0, 1].
        p: Probability of applying the flip.

    Returns:
        Possibly-flipped HWC array.
    '''
    if np.random.rand() < p:
        return image[:, ::-1, :].copy()
    return image


def random_crop(
    image: np.ndarray,
    crop_size: int,
    padding: int = 4,
) -> np.ndarray:
    '''Pad then randomly crop the image to crop_size × crop_size.

    Reflect-padding avoids introducing artificial black borders.  This is the
    most common augmentation for CIFAR-style experiments.

    Args:
        image: HWC float32 array of shape (H, W, C).
        crop_size: Side length of the output square crop.
        padding: Number of pixels to pad on each side before cropping.

    Returns:
        Cropped HWC array of shape (crop_size, crop_size, C).
    '''
    h, w, c = image.shape
    # Reflect-pad: avoids zero padding artefacts
    padded = np.pad(
        image,
        ((padding, padding), (padding, padding), (0, 0)),
        mode="reflect",
    )
    ph, pw = padded.shape[:2]
    top  = np.random.randint(0, ph - crop_size + 1)
    left = np.random.randint(0, pw - crop_size + 1)
    return padded[top: top + crop_size, left: left + crop_size, :]


def random_rotation(
    image: np.ndarray,
    max_degrees: float = 15.0,
) -> np.ndarray:
    '''Rotate image by a uniformly-sampled angle in [-max_degrees, max_degrees].

    Uses a 2-D affine rotation matrix applied via bilinear interpolation.
    Pixels that fall outside the original image boundary are filled with the
    image's per-channel mean to minimise border artefacts.

    Args:
        image: HWC float32 array in [0, 1].
        max_degrees: Maximum rotation angle in degrees.

    Returns:
        Rotated HWC array of the same shape.
    '''
    angle = np.random.uniform(-max_degrees, max_degrees)
    theta = np.radians(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    h, w, c = image.shape
    cx, cy = w / 2.0, h / 2.0

    # Build inverse rotation matrix (source coords → destination)
    rot_mat = np.array([
        [ cos_t, sin_t],
        [-sin_t, cos_t],
    ])

    # Grid of destination pixel coordinates
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    coords = np.stack([xs.ravel() - cx, ys.ravel() - cy], axis=0)  # (2, H*W)
    src_coords = rot_mat @ coords
    src_x = src_coords[0] + cx
    src_y = src_coords[1] + cy

    # Bilinear interpolation
    x0 = np.floor(src_x).astype(np.int32)
    y0 = np.floor(src_y).astype(np.int32)
    x1 = x0 + 1
    y1 = y0 + 1
    dx = src_x - x0
    dy = src_y - y0

    # Clamp indices and mark out-of-bounds
    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)
    x0c = np.clip(x0, 0, w - 1)
    x1c = np.clip(x1, 0, w - 1)
    y0c = np.clip(y0, 0, h - 1)
    y1c = np.clip(y1, 0, h - 1)

    # Interpolate each channel
    output = np.zeros_like(image)
    channel_means = image.mean(axis=(0, 1))
    for ch in range(c):
        top_left     = image[y0c, x0c, ch]
        top_right    = image[y0c, x1c, ch]
        bottom_left  = image[y1c, x0c, ch]
        bottom_right = image[y1c, x1c, ch]
        interp = (
            top_left     * (1 - dx) * (1 - dy)
            + top_right  *      dx  * (1 - dy)
            + bottom_left * (1 - dx) *      dy
            + bottom_right *     dx  *      dy
        )
        # Fill out-of-bounds with channel mean
        interp[~valid] = channel_means[ch]
        output[:, :, ch] = interp.reshape(h, w)

    return np.clip(output, 0.0, 1.0)


def random_affine(
    image: np.ndarray,
    shear_range: float = 10.0,
    scale_range: tuple = (0.9, 1.1),
) -> np.ndarray:
    '''Apply a random affine warp: shear + scale (no rotation/translation).

    Affine transforms are more general than pure rotation and are commonly
    used in document and handwriting recognition to augment character shapes.

    Args:
        image: HWC float32 array in [0, 1].
        shear_range: Maximum shear angle in degrees.
        scale_range: (min_scale, max_scale) for uniform zoom.

    Returns:
        Affine-warped HWC array of the same spatial dimensions.
    '''
    h, w, c = image.shape
    cx, cy = w / 2.0, h / 2.0

    shear = np.radians(np.random.uniform(-shear_range, shear_range))
    scale = np.random.uniform(*scale_range)

    # Inverse affine: dst → src
    affine = np.array([
        [scale,        np.tan(shear)],
        [0.0,          scale        ],
    ])

    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    coords = np.stack([xs.ravel() - cx, ys.ravel() - cy], axis=0)
    src_coords = affine @ coords
    src_x = src_coords[0] + cx
    src_y = src_coords[1] + cy

    x0 = np.floor(src_x).astype(np.int32)
    y0 = np.floor(src_y).astype(np.int32)
    x1, y1 = x0 + 1, y0 + 1
    dx, dy = src_x - x0, src_y - y0

    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)
    x0c = np.clip(x0, 0, w - 1)
    x1c = np.clip(x1, 0, w - 1)
    y0c = np.clip(y0, 0, h - 1)
    y1c = np.clip(y1, 0, h - 1)

    output = np.zeros_like(image)
    channel_means = image.mean(axis=(0, 1))
    for ch in range(c):
        interp = (
            image[y0c, x0c, ch] * (1 - dx) * (1 - dy)
            + image[y0c, x1c, ch] *      dx  * (1 - dy)
            + image[y1c, x0c, ch] * (1 - dx) *      dy
            + image[y1c, x1c, ch] *      dx  *      dy
        )
        interp[~valid] = channel_means[ch]
        output[:, :, ch] = interp.reshape(h, w)

    return np.clip(output, 0.0, 1.0)


print("Geometric transform functions defined.")

### 1.2 Color Jittering

Color jitter perturbs **photometric properties** — brightness, contrast, saturation, and hue — to teach
the model that the same object can appear under different lighting conditions:

$$
I_{\text{bright}} = \text{clip}(I \cdot (1 + \delta_b), 0, 1), \quad \delta_b \sim \mathcal{U}(-\beta, \beta)
$$

Contrast scaling shifts pixel values toward/away from the channel mean:

$$
I_{\text{contrast}} = \text{clip}(\bar{I} + (I - \bar{I}) \cdot \alpha_c, 0, 1)
$$

In [ ]:
# ── Color Jitter ──────────────────────────────────────────────────────────────

def adjust_brightness(image: np.ndarray, factor: float) -> np.ndarray:
    '''Multiply all pixel values by factor and clip to [0, 1].

    Args:
        image: HWC float32 array in [0, 1].
        factor: Multiplicative brightness factor. Values > 1 brighten, < 1 darken.

    Returns:
        Brightness-adjusted HWC array.
    '''
    return np.clip(image * factor, 0.0, 1.0)


def adjust_contrast(image: np.ndarray, factor: float) -> np.ndarray:
    '''Scale contrast around the per-channel mean.

    Args:
        image: HWC float32 array in [0, 1].
        factor: Contrast scale. 1.0 = identity, < 1 reduces, > 1 increases.

    Returns:
        Contrast-adjusted HWC array.
    '''
    mean_vals = image.mean(axis=(0, 1), keepdims=True)  # (1, 1, C)
    return np.clip(mean_vals + (image - mean_vals) * factor, 0.0, 1.0)


def adjust_saturation(image: np.ndarray, factor: float) -> np.ndarray:
    '''Adjust colour saturation by blending toward a grayscale version.

    Saturation = 0 → full grayscale; saturation = 1 → original.

    Args:
        image: HWC float32 RGB array in [0, 1].
        factor: Saturation scale. 0 = grayscale, 1 = original, > 1 = hyper-saturated.

    Returns:
        Saturation-adjusted HWC array.
    '''
    # Luminance weights: ITU-R BT.601 standard
    gray = (
        0.2989 * image[:, :, 0:1]
        + 0.5870 * image[:, :, 1:2]
        + 0.1140 * image[:, :, 2:3]
    )
    return np.clip(gray + (image - gray) * factor, 0.0, 1.0)


def adjust_hue(image: np.ndarray, hue_shift: float) -> np.ndarray:
    '''Shift the hue channel of an RGB image by hue_shift degrees.

    Converts RGB → HSV, rotates the H channel, then converts back.
    Hue shifting requires the color-space conversion implemented in Section 1.3.

    Args:
        image: HWC float32 RGB array in [0, 1].
        hue_shift: Shift in degrees in (-180, 180).

    Returns:
        Hue-shifted HWC RGB array.
    '''
    # Forward declaration — implemented after rgb_to_hsv / hsv_to_rgb below.
    # We call those functions from Section 1.3.
    hsv = rgb_to_hsv(image)  # defined in code-color-spaces below
    hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift / 360.0) % 1.0
    return hsv_to_rgb(hsv)


class ColorJitter:
    '''Randomly apply brightness, contrast, saturation, and hue jitter.

    Each parameter is sampled uniformly from its range and applied in a
    randomised order (following torchvision convention).

    Attributes:
        brightness: Max relative brightness change (0 = no change).
        contrast: Max contrast scale shift.
        saturation: Max saturation scale shift.
        hue: Max hue rotation in degrees (applied in [-hue, +hue]).
    '''

    def __init__(
        self,
        brightness: float = 0.4,
        contrast: float = 0.4,
        saturation: float = 0.4,
        hue: float = 0.1,
    ) -> None:
        '''Store jitter hyperparameters.

        Args:
            brightness: Uniform range [-brightness, brightness] for brightness delta.
            contrast: Uniform range [1-contrast, 1+contrast] for contrast factor.
            saturation: Uniform range [1-saturation, 1+saturation] for saturation factor.
            hue: Max hue rotation in degrees (range [-hue*360, hue*360]).
        '''
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue

    def __call__(self, image: np.ndarray) -> np.ndarray:
        '''Apply randomised color jitter to image.

        Args:
            image: HWC float32 RGB array in [0, 1].

        Returns:
            Jittered HWC array.
        '''
        # Random order of transforms
        order = np.random.permutation(4)
        img = image.copy()
        for op in order:
            if op == 0 and self.brightness > 0:
                factor = 1.0 + np.random.uniform(-self.brightness, self.brightness)
                img = adjust_brightness(img, factor)
            elif op == 1 and self.contrast > 0:
                factor = 1.0 + np.random.uniform(-self.contrast, self.contrast)
                img = adjust_contrast(img, factor)
            elif op == 2 and self.saturation > 0:
                factor = 1.0 + np.random.uniform(-self.saturation, self.saturation)
                img = adjust_saturation(img, factor)
            elif op == 3 and self.hue > 0:
                shift = np.random.uniform(-self.hue * 360, self.hue * 360)
                img = adjust_hue(img, shift)
        return img


print("ColorJitter class defined.")

### 1.3 Color Space Conversions: RGB ↔ HSV ↔ LAB

Understanding color spaces matters for:
- **Hue shifting** requires working in HSV, where hue is a single channel.
- **Perceptual uniformity** (LAB) is critical in medical imaging and satellite analysis.
- **Normalization strategy** depends on which color space the network sees.

**RGB → HSV:**
Let $M = \max(R,G,B)$, $m = \min(R,G,B)$, $\Delta = M - m$.
$$H = \begin{cases} 0 & \Delta=0 \\ 60\cdot\frac{G-B}{\Delta} \mod 360 & M=R \\ 60\cdot\frac{B-R}{\Delta}+120 & M=G \\ 60\cdot\frac{R-G}{\Delta}+240 & M=B \end{cases}, \quad S = \frac{\Delta}{M}, \quad V = M$$

In [ ]:
# ── Color Space Conversions ───────────────────────────────────────────────────

def rgb_to_hsv(image: np.ndarray) -> np.ndarray:
    '''Convert an RGB image to HSV (Hue–Saturation–Value).

    All three output channels are in [0, 1].  Hue is normalised so that
    0 = red, 1/3 = green, 2/3 = blue (i.e., degrees / 360).

    Args:
        image: HWC float32 RGB array in [0, 1].

    Returns:
        HWC float32 HSV array where H, S, V are all in [0, 1].
    '''
    r, g, b = image[:, :, 0], image[:, :, 1], image[:, :, 2]
    max_val = np.maximum(np.maximum(r, g), b)
    min_val = np.minimum(np.minimum(r, g), b)
    delta   = max_val - min_val

    # Value
    v = max_val

    # Saturation
    s = np.where(max_val > 0, delta / max_val, 0.0)

    # Hue (normalised to [0, 1])
    h = np.zeros_like(r)
    eps = 1e-8
    mask_r = (max_val == r) & (delta > 0)
    mask_g = (max_val == g) & (delta > 0)
    mask_b = (max_val == b) & (delta > 0)

    h[mask_r] = ((g[mask_r] - b[mask_r]) / (delta[mask_r] + eps)) % 6.0
    h[mask_g] = ((b[mask_g] - r[mask_g]) / (delta[mask_g] + eps)) + 2.0
    h[mask_b] = ((r[mask_b] - g[mask_b]) / (delta[mask_b] + eps)) + 4.0
    h = h / 6.0  # normalise to [0, 1]
    h = np.clip(h, 0.0, 1.0)

    return np.stack([h, s, v], axis=-1)


def hsv_to_rgb(image: np.ndarray) -> np.ndarray:
    '''Convert an HSV image back to RGB.

    Args:
        image: HWC float32 HSV array where H, S, V are all in [0, 1].

    Returns:
        HWC float32 RGB array in [0, 1].
    '''
    h, s, v = image[:, :, 0], image[:, :, 1], image[:, :, 2]
    h6 = h * 6.0
    i  = np.floor(h6).astype(np.int32) % 6
    f  = h6 - np.floor(h6)
    p  = v * (1.0 - s)
    q  = v * (1.0 - f * s)
    t  = v * (1.0 - (1.0 - f) * s)

    r = np.select([i==0, i==1, i==2, i==3, i==4, i==5], [v, q, p, p, t, v])
    g = np.select([i==0, i==1, i==2, i==3, i==4, i==5], [t, v, v, q, p, p])
    b = np.select([i==0, i==1, i==2, i==3, i==4, i==5], [p, p, t, v, v, q])

    return np.clip(np.stack([r, g, b], axis=-1), 0.0, 1.0)


def rgb_to_lab(image: np.ndarray) -> np.ndarray:
    '''Convert RGB to CIE L*a*b* (D65 illuminant, sRGB primaries).

    L* is the perceptual lightness (0 = black, 100 = white).  a* encodes
    green-red and b* encodes blue-yellow.  LAB is designed to be perceptually
    uniform — Euclidean distance in LAB correlates with human color difference.

    Args:
        image: HWC float32 RGB array in [0, 1].

    Returns:
        HWC float32 LAB array.  L in [0, 100], a and b in approximately [-128, 127].
    '''
    # Step 1: sRGB → linear RGB (undo gamma)
    linear = np.where(
        image <= 0.04045,
        image / 12.92,
        ((image + 0.055) / 1.055) ** 2.4,
    )

    # Step 2: linear RGB → XYZ (D65)
    mat = np.array([
        [0.4124564, 0.3575761, 0.1804375],
        [0.2126729, 0.7151522, 0.0721750],
        [0.0193339, 0.1191920, 0.9503041],
    ])
    xyz = linear @ mat.T  # (H, W, 3) @ (3, 3)

    # Step 3: normalise by D65 white point
    xyz = xyz / np.array([0.95047, 1.00000, 1.08883])

    # Step 4: apply f() function
    delta = 6.0 / 29.0
    fx = np.where(xyz > delta**3, np.cbrt(xyz), xyz / (3 * delta**2) + 4.0 / 29.0)

    # Step 5: XYZ → L*a*b*
    L = 116.0 * fx[:, :, 1] - 16.0
    a = 500.0 * (fx[:, :, 0] - fx[:, :, 1])
    b_ch = 200.0 * (fx[:, :, 1] - fx[:, :, 2])

    return np.stack([L, a, b_ch], axis=-1)


# ── Validation: round-trip RGB → HSV → RGB ────────────────────────────────────
rng_val = np.random.default_rng(SEED)
test_img = rng_val.random((8, 8, 3)).astype(np.float32)
reconstructed = hsv_to_rgb(rgb_to_hsv(test_img))
max_err = np.abs(test_img - reconstructed).max()
print(f"RGB → HSV → RGB max pixel error: {max_err:.2e}  (should be < 1e-5)")
assert max_err < 1e-4, "Round-trip error too large"
print("Color space conversion validated.")

### Visualising Geometric & Color Augmentations

Apply each transform to a single CIFAR-10 image to confirm visual correctness.

In [ ]:
# ── Visualise all augmentations on one CIFAR-10 image ─────────────────────────
np.random.seed(SEED)

sample_tensor, sample_label = full_train_raw[7]  # a bird
sample_np = sample_tensor.permute(1, 2, 0).numpy()  # CHW → HWC

jitter = ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.15)

aug_images = {
    "Original": sample_np,
    "H-Flip": horizontal_flip(sample_np, p=1.0),
    "RandomCrop": random_crop(sample_np, crop_size=28, padding=4),
    "Rotation 15°": random_rotation(sample_np, max_degrees=15),
    "Affine": random_affine(sample_np),
    "Brightness×1.6": adjust_brightness(sample_np, 1.6),
    "Contrast×2": adjust_contrast(sample_np, 2.0),
    "Saturation 0": adjust_saturation(sample_np, 0.0),
    "ColorJitter": jitter(sample_np),
    "HSV H-channel": rgb_to_hsv(sample_np)[:, :, 0:1].repeat(3, axis=2),
    "LAB L-channel": (rgb_to_lab(sample_np)[:, :, 0:1] / 100.0).repeat(3, axis=2),
}

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
axes = axes.ravel()
for ax in axes:
    ax.axis("off")
for i, (title, img) in enumerate(aug_images.items()):
    axes[i].imshow(np.clip(img, 0, 1))
    axes[i].set_title(title, fontsize=9)
    axes[i].axis("off")

plt.suptitle(
    f'Augmentation Gallery — CIFAR-10 "{CIFAR10_CLASSES[sample_label]}"',
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.show()

### 1.4 Advanced Augmentation: Mixup & CutMix

**Mixup** creates a virtual training example by taking a **linear combination** of two samples:
$$\tilde{x} = \lambda x_i + (1-\lambda) x_j, \quad \tilde{y} = \lambda y_i + (1-\lambda) y_j, \quad \lambda \sim \text{Beta}(\alpha, \alpha)$$

**CutMix** replaces a rectangular patch of one image with a patch from another,
proportionally mixing the labels based on the patch area:
$$\tilde{x} = \mathbf{M} \odot x_i + (1-\mathbf{M}) \odot x_j, \quad \tilde{y} = \lambda y_i + (1-\lambda) y_j$$
where $\mathbf{M}$ is a binary mask and $\lambda = 1 - \text{area}(\text{patch})/\text{area}(\text{image})$.

Both techniques act as **label-smoothing regularizers** that reduce overconfident predictions.

In [ ]:
# ── Mixup & CutMix ────────────────────────────────────────────────────────────

def mixup_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = 0.4,
    num_classes: int = 10,
) -> tuple[torch.Tensor, torch.Tensor]:
    '''Apply Mixup to a batch of images and one-hot labels.

    Samples lambda from Beta(alpha, alpha).  Shuffles the batch and linearly
    interpolates image tensors and soft labels simultaneously.

    Args:
        images: Float tensor of shape (N, C, H, W) in [0, 1].
        labels: Integer tensor of shape (N,) with class indices.
        alpha: Beta distribution concentration parameter.
        num_classes: Total number of classes for one-hot encoding.

    Returns:
        Tuple of (mixed_images, mixed_labels) where mixed_labels are soft
        labels of shape (N, num_classes).
    '''
    batch_size = images.size(0)
    lam = float(np.random.beta(alpha, alpha))

    # Shuffle indices for the second image in each pair
    perm = torch.randperm(batch_size, device=images.device)

    # One-hot encode labels
    labels_onehot = torch.zeros(batch_size, num_classes, device=images.device)
    labels_onehot.scatter_(1, labels.unsqueeze(1), 1.0)
    labels_perm_onehot = labels_onehot[perm]

    mixed_images = lam * images + (1.0 - lam) * images[perm]
    mixed_labels = lam * labels_onehot + (1.0 - lam) * labels_perm_onehot

    return mixed_images, mixed_labels


def cutmix_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = 1.0,
    num_classes: int = 10,
) -> tuple[torch.Tensor, torch.Tensor]:
    '''Apply CutMix to a batch of images and one-hot labels.

    Samples a random rectangular box whose area fraction is lambda_box.  Pastes
    a patch from image j into image i, mixing labels by (1 - lambda_box).

    Args:
        images: Float tensor of shape (N, C, H, W) in [0, 1].
        labels: Integer tensor of shape (N,) with class indices.
        alpha: Beta concentration for sampling lambda_box.
        num_classes: Total number of classes.

    Returns:
        Tuple of (cut_images, mixed_labels) with soft labels of shape (N, num_classes).
    '''
    batch_size, c, h, w = images.shape
    lam = float(np.random.beta(alpha, alpha))

    # Bounding box with area ~ (1 - lam)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(h * cut_ratio)
    cut_w = int(w * cut_ratio)

    cx = np.random.randint(w)
    cy = np.random.randint(h)
    x1 = np.clip(cx - cut_w // 2, 0, w)
    x2 = np.clip(cx + cut_w // 2, 0, w)
    y1 = np.clip(cy - cut_h // 2, 0, h)
    y2 = np.clip(cy + cut_h // 2, 0, h)

    # Actual lambda based on clipped box
    lam_actual = 1.0 - (x2 - x1) * (y2 - y1) / (h * w)

    perm = torch.randperm(batch_size, device=images.device)
    mixed_images = images.clone()
    mixed_images[:, :, y1:y2, x1:x2] = images[perm, :, y1:y2, x1:x2]

    # Soft labels
    labels_onehot = torch.zeros(batch_size, num_classes, device=images.device)
    labels_onehot.scatter_(1, labels.unsqueeze(1), 1.0)
    labels_perm_onehot = labels_onehot[perm]
    mixed_labels = lam_actual * labels_onehot + (1.0 - lam_actual) * labels_perm_onehot

    return mixed_images, mixed_labels


# ── Visualise Mixup and CutMix ─────────────────────────────────────────────────
imgs_sample = torch.stack([
    full_train_raw[i][0] for i in [0, 1, 2, 3]
])
lbls_sample = torch.tensor([full_train_raw[i][1] for i in [0, 1, 2, 3]])

np.random.seed(SEED)
mixed_imgs, mixed_lbls = mixup_batch(imgs_sample, lbls_sample, alpha=0.4)
cut_imgs, cut_lbls     = cutmix_batch(imgs_sample, lbls_sample, alpha=1.0)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
row_titles = ["Original", "After Mixup", "After CutMix"]
for col in range(4):
    axes[0, col].imshow(imgs_sample[col].permute(1, 2, 0))
    axes[0, col].set_title(CIFAR10_CLASSES[lbls_sample[col].item()], fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(mixed_imgs[col].permute(1, 2, 0).clamp(0, 1))
    top_class = mixed_lbls[col].argmax().item()
    axes[1, col].set_title(f"{CIFAR10_CLASSES[top_class]} (soft)", fontsize=9)
    axes[1, col].axis("off")

    axes[2, col].imshow(cut_imgs[col].permute(1, 2, 0).clamp(0, 1))
    top_class_c = cut_lbls[col].argmax().item()
    axes[2, col].set_title(f"{CIFAR10_CLASSES[top_class_c]} (soft)", fontsize=9)
    axes[2, col].axis("off")

for row, title in enumerate(row_titles):
    axes[row, 0].set_ylabel(title, fontsize=10, fontweight="bold", rotation=90, labelpad=5)

plt.suptitle("Mixup vs CutMix on CIFAR-10", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Mixup and CutMix validated.")

---
## Part 2 — Augmentation Pipeline & torchvision Validation

We assemble the individual primitives into a unified `AugmentationPipeline` class
that wraps NumPy operations and integrates with PyTorch's DataLoader via a custom
Dataset wrapper.  We then validate our from-scratch transforms against torchvision
equivalents on the same images.

In [ ]:
# ── AugmentationPipeline: assembles primitives into a DataLoader-compatible form

class AugmentationPipeline:
    '''Configurable augmentation pipeline for image classification.

    Operates on HWC float32 NumPy arrays in [0, 1].  Applies a sequence of
    transforms controlled by boolean flags and strength parameters.

    Attributes:
        do_hflip: Whether to apply random horizontal flip.
        do_crop: Whether to apply random crop with padding.
        crop_size: Target crop size (pixels).
        crop_padding: Padding pixels before random crop.
        do_rotation: Whether to apply random rotation.
        max_rotation: Maximum rotation angle in degrees.
        do_jitter: Whether to apply color jitter.
        jitter: ColorJitter instance.
        mean: Per-channel normalisation mean as (R, G, B) tuple.
        std: Per-channel normalisation std as (R, G, B) tuple.
    '''

    def __init__(
        self,
        do_hflip: bool = True,
        do_crop: bool = True,
        crop_size: int = 32,
        crop_padding: int = 4,
        do_rotation: bool = False,
        max_rotation: float = 15.0,
        do_jitter: bool = False,
        brightness: float = 0.4,
        contrast: float = 0.4,
        saturation: float = 0.4,
        hue: float = 0.1,
        mean: tuple = CIFAR10_MEAN,
        std: tuple = CIFAR10_STD,
    ) -> None:
        '''Initialise the augmentation pipeline with per-transform flags.

        Args:
            do_hflip: Enable random horizontal flip.
            do_crop: Enable random crop with padding.
            crop_size: Output spatial size after cropping.
            crop_padding: Pixels to pad before sampling the crop.
            do_rotation: Enable random rotation.
            max_rotation: Maximum absolute rotation angle in degrees.
            do_jitter: Enable ColorJitter.
            brightness: ColorJitter brightness strength.
            contrast: ColorJitter contrast strength.
            saturation: ColorJitter saturation strength.
            hue: ColorJitter hue strength.
            mean: Per-channel mean for final normalisation.
            std: Per-channel standard deviation for final normalisation.
        '''
        self.do_hflip    = do_hflip
        self.do_crop     = do_crop
        self.crop_size   = crop_size
        self.crop_padding = crop_padding
        self.do_rotation = do_rotation
        self.max_rotation = max_rotation
        self.do_jitter   = do_jitter
        self.jitter      = ColorJitter(brightness, contrast, saturation, hue)
        self.mean        = np.array(mean, dtype=np.float32)
        self.std         = np.array(std,  dtype=np.float32)

    def __call__(self, image_tensor: torch.Tensor) -> torch.Tensor:
        '''Apply the configured augmentation sequence to a CHW image tensor.

        Converts CHW → HWC for NumPy operations, applies transforms, then
        converts back to CHW and normalises.

        Args:
            image_tensor: Float32 CHW tensor in [0, 1].

        Returns:
            Augmented and normalised CHW float32 tensor.
        '''
        img = image_tensor.permute(1, 2, 0).numpy()  # CHW → HWC

        if self.do_hflip:
            img = horizontal_flip(img)
        if self.do_crop:
            img = random_crop(img, self.crop_size, self.crop_padding)
        if self.do_rotation:
            img = random_rotation(img, self.max_rotation)
        if self.do_jitter:
            img = self.jitter(img)

        # Normalise: (img - mean) / std
        img = (img - self.mean) / self.std

        return torch.from_numpy(img.transpose(2, 0, 1)).float()  # HWC → CHW


# ── Validate: compare scratch hflip+crop normalisation with torchvision ───────
np.random.seed(42)
torch.manual_seed(42)

scratch_pipeline = AugmentationPipeline(
    do_hflip=True, do_crop=True, crop_size=32, crop_padding=4,
    do_rotation=False, do_jitter=False,
)

tv_pipeline = T.Compose([
    T.RandomHorizontalFlip(p=1.0),  # force flip for deterministic test
    T.RandomCrop(32, padding=4),
    T.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

# Apply scratch pipeline (deterministic: force flip, manual crop position)
test_tensor, _ = full_train_raw[0]  # bird image, CHW in [0, 1]

# Normalise only (no stochastic transforms) — compare normalisation correctness
norm_only = AugmentationPipeline(
    do_hflip=False, do_crop=False, do_rotation=False, do_jitter=False,
)
scratch_normalised = norm_only(test_tensor)
tv_normalised = T.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)(test_tensor)

max_diff = (scratch_normalised - tv_normalised).abs().max().item()
print(f"Normalisation max pixel diff vs torchvision: {max_diff:.2e}  (should be < 1e-5)")
assert max_diff < 1e-4
print("AugmentationPipeline validated against torchvision normalisation.")

### 2.1 AutoAugment Policy Concepts

**AutoAugment** (Cubuk et al., 2019) searches for optimal augmentation policies using reinforcement
learning.  A **policy** is a sequence of sub-policies; each sub-policy contains two operations
$(op_1, p_1, m_1)$ and $(op_2, p_2, m_2)$ where $p$ is the probability and $m$ is the magnitude.

We implement a **simplified policy sampler** that demonstrates the concept without the RL search:

In [ ]:
# ── AutoAugment Policy Concept ────────────────────────────────────────────────

# An operation is a (function, max_magnitude) pair
# The magnitude parameter scales strength from 0 (no-op) to 1 (max strength)

def _brightness_op(image: np.ndarray, magnitude: float) -> np.ndarray:
    '''Apply brightness jitter scaled by magnitude in [0, 1].

    Args:
        image: HWC float32 array in [0, 1].
        magnitude: Augmentation strength in [0, 1].

    Returns:
        Brightness-adjusted HWC array.
    '''
    factor = 1.0 + magnitude * np.random.choice([-1, 1]) * 0.9
    return adjust_brightness(image, max(0.1, factor))


def _contrast_op(image: np.ndarray, magnitude: float) -> np.ndarray:
    '''Apply contrast jitter scaled by magnitude.

    Args:
        image: HWC float32 array in [0, 1].
        magnitude: Augmentation strength in [0, 1].

    Returns:
        Contrast-adjusted HWC array.
    '''
    factor = 1.0 + magnitude * np.random.choice([-1, 1]) * 0.9
    return adjust_contrast(image, max(0.1, factor))


def _rotate_op(image: np.ndarray, magnitude: float) -> np.ndarray:
    '''Apply rotation scaled by magnitude (max 30 degrees).

    Args:
        image: HWC float32 array in [0, 1].
        magnitude: Augmentation strength in [0, 1].

    Returns:
        Rotated HWC array.
    '''
    max_deg = magnitude * 30.0
    return random_rotation(image, max_deg)


def _hflip_op(image: np.ndarray, magnitude: float) -> np.ndarray:  # noqa: ARG001
    '''Apply horizontal flip (magnitude unused).

    Args:
        image: HWC float32 array in [0, 1].
        magnitude: Unused (flip is binary).

    Returns:
        Flipped HWC array.
    '''
    return horizontal_flip(image, p=1.0)


# CIFAR-10 AutoAugment policy (simplified — 5 representative sub-policies)
AUTOAUGMENT_POLICY: list = [
    # (op_fn_1, prob_1, mag_1, op_fn_2, prob_2, mag_2)
    (_brightness_op, 0.4, 0.6, _contrast_op,  0.6, 0.4),
    (_hflip_op,      0.8, 1.0, _rotate_op,     0.7, 0.3),
    (_rotate_op,     0.6, 0.5, _brightness_op, 0.5, 0.5),
    (_contrast_op,   0.5, 0.7, _hflip_op,      0.4, 1.0),
    (_brightness_op, 0.3, 0.8, _rotate_op,     0.5, 0.6),
]


def apply_autoaugment_policy(
    image: np.ndarray,
    policy: list,
) -> np.ndarray:
    '''Apply one randomly selected sub-policy from an AutoAugment policy.

    Randomly selects a sub-policy and applies each of its two operations
    independently with the specified probability.

    Args:
        image: HWC float32 array in [0, 1].
        policy: List of sub-policies, each a tuple of
            (op1, p1, m1, op2, p2, m2).

    Returns:
        Augmented HWC array.
    '''
    sub = policy[np.random.randint(len(policy))]
    op1_fn, p1, m1, op2_fn, p2, m2 = sub
    img = image.copy()
    if np.random.rand() < p1:
        img = op1_fn(img, m1)
    if np.random.rand() < p2:
        img = op2_fn(img, m2)
    return img


# Visualise AutoAugment on 8 applications of the same image
np.random.seed(SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    aa_img = apply_autoaugment_policy(sample_np, AUTOAUGMENT_POLICY)
    ax.imshow(np.clip(aa_img, 0, 1))
    ax.set_title(f"AutoAugment #{i+1}", fontsize=9)
    ax.axis("off")
plt.suptitle("AutoAugment: 8 Stochastic Applications", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("AutoAugment policy visualised.")

---
## Part 3 — Applying Augmentation to CIFAR-10 Training

We train a small CNN under four augmentation configurations and compare val accuracy:

| Config | Transforms |
|--------|------------|
| Baseline | Normalisation only |
| Standard | H-Flip + RandomCrop + Normalise |
| Color | Standard + ColorJitter |
| Mixup | Standard + Mixup during training |

In [ ]:
# ── Small CNN for CIFAR-10 ────────────────────────────────────────────────────

class SmallCNN(nn.Module):
    '''Lightweight 3-block convolutional network for CIFAR-10.

    Architecture: 3 × (Conv-BN-ReLU-MaxPool) → GlobalAvgPool → Linear.
    Designed for quick training (~5 epochs) to measure augmentation effects.

    Attributes:
        features: Sequential feature extractor.
        classifier: Final linear projection to num_classes.
    '''

    def __init__(self, num_classes: int = 10) -> None:
        '''Build convolutional feature extractor and classifier head.

        Args:
            num_classes: Number of output classes.
        '''
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),               # 32 → 16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),               # 16 → 8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),       # 8 → 1
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass: extract features then classify.

        Args:
            x: Input tensor of shape (N, 3, H, W).

        Returns:
            Logit tensor of shape (N, num_classes).
        '''
        features = self.features(x).squeeze(-1).squeeze(-1)  # (N, 128)
        return self.classifier(features)


n_params = sum(p.numel() for p in SmallCNN().parameters())
print(f"SmallCNN parameters: {n_params:,}")

In [ ]:
# ── Build DataLoaders for each augmentation config ───────────────────────────

# Official train/val split (torchvision provides train=True/False)
# We use the official test split as a held-out val set for speed

def build_loaders(aug_pipeline: Callable, batch_size: int = BATCH_SIZE) -> tuple:
    '''Create train and validation DataLoaders with the given augmentation.

    The augmentation pipeline is applied only to the training set; the val
    set always receives normalisation-only preprocessing.

    Args:
        aug_pipeline: Callable transform applied to each training image.
        batch_size: Mini-batch size.

    Returns:
        Tuple of (train_loader, val_loader).
    '''
    norm_only_pipeline = AugmentationPipeline(
        do_hflip=False, do_crop=False, do_rotation=False, do_jitter=False,
    )

    train_dataset = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=False,
        transform=aug_pipeline,
    )
    val_dataset = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=False, download=False,
        transform=norm_only_pipeline,
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader


# Define four augmentation configurations
baseline_pipe = AugmentationPipeline(
    do_hflip=False, do_crop=False, do_rotation=False, do_jitter=False,
)
standard_pipe = AugmentationPipeline(
    do_hflip=True, do_crop=True, do_rotation=False, do_jitter=False,
)
color_pipe = AugmentationPipeline(
    do_hflip=True, do_crop=True, do_rotation=False, do_jitter=True,
    brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05,
)
rotation_pipe = AugmentationPipeline(
    do_hflip=True, do_crop=True, do_rotation=True, max_rotation=10.0,
    do_jitter=False,
)

configs = {
    "Baseline": (baseline_pipe, False),   # (pipeline, use_mixup)
    "Standard": (standard_pipe, False),
    "Color": (color_pipe, False),
    "Mixup": (standard_pipe, True),
}

print(f"Defined {len(configs)} augmentation configurations.")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

def train_one_epoch_aug(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    use_mixup: bool = False,
) -> tuple[float, float]:
    '''Train for one epoch, optionally applying Mixup.

    When Mixup is active, labels are soft (one-hot float) and the loss uses
    ``nn.CrossEntropyLoss`` which supports soft label targets natively.

    Args:
        model: Neural network to train.
        loader: Training DataLoader.
        optimizer: Parameter optimizer.
        criterion: Loss function (expects logits and soft labels).
        device: Compute device.
        use_mixup: Whether to apply Mixup augmentation on each batch.

    Returns:
        Tuple of (average_loss, accuracy) for the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if use_mixup:
            images, soft_labels = mixup_batch(
                images, labels, alpha=0.4, num_classes=NUM_CLASSES
            )
            soft_labels = soft_labels.to(device)
        else:
            soft_labels = None

        optimizer.zero_grad()
        logits = model(images)

        if soft_labels is not None:
            # KL divergence between soft targets and predicted log-probabilities
            log_probs = F.log_softmax(logits, dim=1)
            loss = -(soft_labels * log_probs).sum(dim=1).mean()
        else:
            loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = logits.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    return running_loss / total, correct / total


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate model on a validation DataLoader.

    Args:
        model: Neural network to evaluate.
        loader: Validation DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = logits.max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

    return running_loss / total, correct / total


# ── Run training for all four configs ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
all_histories: dict = {}

for config_name, (pipe, use_mixup) in configs.items():
    print(f"\n{'='*60}")
    print(f"Config: {config_name}  (Mixup={use_mixup})")
    print('='*60)

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = SmallCNN(num_classes=NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    train_loader, val_loader = build_loaders(pipe)

    history: dict = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch_aug(
            model, train_loader, optimizer, criterion, device, use_mixup
        )
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2%} | "
            f"Time: {elapsed:.1f}s"
        )

    all_histories[config_name] = history

print("\nAll training runs complete.")

---
## Part 4 — Evaluation & Analysis

In [ ]:
# ── Summary results table ─────────────────────────────────────────────────────
results_rows = []
for config_name, hist in all_histories.items():
    best_val_acc  = max(hist["val_acc"])
    final_val_acc = hist["val_acc"][-1]
    final_train_loss = hist["train_loss"][-1]
    final_val_loss   = hist["val_loss"][-1]
    gap = final_train_loss - final_val_loss
    results_rows.append({
        "Config": config_name,
        "Best Val Acc": f"{best_val_acc:.2%}",
        "Final Val Acc": f"{final_val_acc:.2%}",
        "Final Train Loss": f"{final_train_loss:.4f}",
        "Final Val Loss": f"{final_val_loss:.4f}",
        "Train-Val Loss Gap": f"{gap:.4f}",
    })

results_df = pd.DataFrame(results_rows)
print("Augmentation Config Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# ── Plot training curves for all configs ─────────────────────────────────────
epochs_x = list(range(1, NUM_EPOCHS + 1))
colors = ["steelblue", "coral", "green", "purple"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (config_name, hist) in enumerate(all_histories.items()):
    color = colors[i]
    axes[0].plot(epochs_x, hist["train_loss"], linestyle="--",
                 color=color, alpha=0.6, label=f"{config_name} (train)")
    axes[0].plot(epochs_x, hist["val_loss"], linestyle="-",
                 color=color, label=f"{config_name} (val)")

    axes[1].plot(epochs_x, [v * 100 for v in hist["val_acc"]],
                 marker="o", color=color, label=config_name)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Train vs Val Loss", fontweight="bold")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Accuracy (%)")
axes[1].set_title("Val Accuracy by Augmentation Config", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Effect of Augmentation Strategy on CIFAR-10 Training",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Overfitting analysis: train-val loss gap across configs ───────────────────
# A larger train-val gap indicates more overfitting.
# Augmentation should reduce the gap by regularising the model.

fig, ax = plt.subplots(figsize=(8, 4))
config_names = list(all_histories.keys())
bar_colors = colors[:len(config_names)]

gaps = [
    all_histories[cfg]["train_loss"][-1] - all_histories[cfg]["val_loss"][-1]
    for cfg in config_names
]
bars = ax.bar(config_names, gaps, color=bar_colors, edgecolor="white", width=0.5)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Train Loss − Val Loss (final epoch)", fontsize=11)
ax.set_title("Generalisation Gap by Augmentation Config\n"
             "(negative = val loss higher than train, i.e., moderate variance)",
             fontsize=11, fontweight="bold")
ax.grid(True, axis="y", alpha=0.3)

for bar, gap_val in zip(bars, gaps):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
        f"{gap_val:+.4f}", ha="center", va="bottom", fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
# ── Color space analysis: compare RGB vs HSV vs LAB channel statistics ────────
# Sample 500 CIFAR-10 training images and compare channel distributions
sample_count = 500
rgb_channels: list = [[], [], []]
hsv_channels: list = [[], [], []]
lab_l_values: list = []

for i in range(sample_count):
    img_t, _ = full_train_raw[i]
    img_np = img_t.permute(1, 2, 0).numpy()  # HWC in [0, 1]
    for ch in range(3):
        rgb_channels[ch].append(img_np[:, :, ch].mean())
    hsv = rgb_to_hsv(img_np)
    for ch in range(3):
        hsv_channels[ch].append(hsv[:, :, ch].mean())
    lab = rgb_to_lab(img_np)
    lab_l_values.append(lab[:, :, 0].mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ch_colors = ["red", "green", "blue"]
ch_labels_rgb = ["R", "G", "B"]
ch_labels_hsv = ["H (hue)", "S (sat)", "V (value)"]

for ch in range(3):
    axes[0].hist(rgb_channels[ch], bins=40, alpha=0.6,
                 color=ch_colors[ch], label=ch_labels_rgb[ch])
axes[0].set_title("RGB Channel Means", fontweight="bold")
axes[0].set_xlabel("Mean intensity")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for ch in range(3):
    axes[1].hist(hsv_channels[ch], bins=40, alpha=0.6,
                 color=ch_colors[ch], label=ch_labels_hsv[ch])
axes[1].set_title("HSV Channel Means", fontweight="bold")
axes[1].set_xlabel("Mean value (normalised)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(lab_l_values, bins=40, color="goldenrod", alpha=0.8)
axes[2].set_title("LAB L* (Lightness) Means", fontweight="bold")
axes[2].set_xlabel("Mean L* (0–100)")
axes[2].grid(True, alpha=0.3)

plt.suptitle("CIFAR-10: Channel Statistics Across Color Spaces (500 images)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nRGB mean R={np.mean(rgb_channels[0]):.3f}, "
      f"G={np.mean(rgb_channels[1]):.3f}, B={np.mean(rgb_channels[2]):.3f}")
print(f"HSV mean H={np.mean(hsv_channels[0]):.3f}, "
      f"S={np.mean(hsv_channels[1]):.3f}, V={np.mean(hsv_channels[2]):.3f}")
print(f"LAB mean L*={np.mean(lab_l_values):.1f}")

In [ ]:
# ── Augmentation diversity: pixel-level variance across 20 augmented versions ─
# High variance across augmented copies indicates the pipeline is diverse.
# Low variance indicates redundant or too-weak augmentation.

np.random.seed(SEED)
num_copies = 20
aug_stack = np.zeros((num_copies, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

for k in range(num_copies):
    aug_stack[k] = standard_pipe(test_tensor).permute(1, 2, 0).numpy()

# Per-pixel variance (H, W, 3) → scalar
pixel_var = aug_stack.var(axis=0)  # (H, W, 3)
mean_var_per_config = {"Baseline": 0.0}  # deterministic → zero variance

configs_to_measure = {
    "Standard": standard_pipe,
    "Color": color_pipe,
    "Rotation": rotation_pipe,
}
for cfg_name, pipe in configs_to_measure.items():
    stack = np.zeros((num_copies, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    for k in range(num_copies):
        stack[k] = pipe(test_tensor).permute(1, 2, 0).numpy()
    mean_var_per_config[cfg_name] = float(stack.var(axis=0).mean())

fig, ax = plt.subplots(figsize=(7, 4))
cfg_labels = list(mean_var_per_config.keys())
var_values  = list(mean_var_per_config.values())
ax.bar(cfg_labels, var_values, color=colors[:len(cfg_labels)],
       edgecolor="white", width=0.5)
ax.set_ylabel("Mean pixel variance across 20 augmented copies", fontsize=11)
ax.set_title("Augmentation Diversity (higher = more diverse views)",
             fontsize=12, fontweight="bold")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Mean pixel variance per config:")
for cfg_name, var_val in mean_var_per_config.items():
    print(f"  {cfg_name:10s}: {var_val:.6f}")

In [ ]:
# ── Normalisation strategy comparison ────────────────────────────────────────
# Compare three normalisation schemes on 200 CIFAR-10 images:
#   1. ImageNet statistics (standard transfer learning)
#   2. CIFAR-10 dataset statistics
#   3. Per-image zero-mean unit-variance

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CIFAR_MEAN    = np.array(CIFAR10_MEAN, dtype=np.float32)
CIFAR_STD     = np.array(CIFAR10_STD,  dtype=np.float32)

def normalise_global(img: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    '''Normalise using dataset-level mean and std.

    Args:
        img: HWC float32 array in [0, 1].
        mean: Per-channel mean of shape (3,).
        std: Per-channel std of shape (3,).

    Returns:
        Normalised HWC float32 array.
    '''
    return (img - mean) / std


def normalise_per_image(img: np.ndarray) -> np.ndarray:
    '''Normalise each image independently to zero mean, unit variance.

    Args:
        img: HWC float32 array in [0, 1].

    Returns:
        Per-image normalised HWC float32 array.
    '''
    mu  = img.mean()
    sig = img.std() + 1e-8
    return (img - mu) / sig


num_eval = 200
norms: dict = {
    "ImageNet stats": [],
    "CIFAR-10 stats": [],
    "Per-image": [],
}

for i in range(num_eval):
    img_t, _ = full_train_raw[i]
    img_np = img_t.permute(1, 2, 0).numpy()
    norms["ImageNet stats"].append(normalise_global(img_np, IMAGENET_MEAN, IMAGENET_STD))
    norms["CIFAR-10 stats"].append(normalise_global(img_np, CIFAR_MEAN, CIFAR_STD))
    norms["Per-image"].append(normalise_per_image(img_np))

print("Normalisation statistics (mean ± std of normalised pixel values):")
print(f"{'Strategy':20s}  {'Mean':>10s}  {'Std':>10s}  {'Min':>10s}  {'Max':>10s}")
print("-" * 60)
for strategy, imgs_list in norms.items():
    all_pixels = np.concatenate([img.ravel() for img in imgs_list])
    print(
        f"{strategy:20s}  "
        f"{all_pixels.mean():>10.4f}  "
        f"{all_pixels.std():>10.4f}  "
        f"{all_pixels.min():>10.4f}  "
        f"{all_pixels.max():>10.4f}"
    )

print("\nInsight: CIFAR-10 statistics centre the distribution closer to zero.")
print("Per-image normalisation is most aggressive but loses inter-image brightness info.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Augmentation is regularisation**: geometric and color transforms expand the effective training set
   by teaching the model invariances — horizontal symmetry (flip), translation (crop), lighting variation
   (color jitter).  Each invariance must be *domain-appropriate*: hflip makes sense for CIFAR-10 but
   not for digit recognition (flipped '6' ≠ '6').

2. **Geometric transforms first**: random crop + horizontal flip are the two highest-impact augmentations
   for CIFAR-10.  They consistently close the train-val generalisation gap more than color jitter alone.

3. **Mixup/CutMix reduce overconfidence**: by training on convex combinations of examples, they act as
   label-smoothing regularisers.  This is most beneficial when training loss reaches near-zero before
   validation loss plateaus (classic overfitting regime).

4. **Color spaces encode different information**: RGB is convenient for network inputs; HSV decouples
   hue from brightness (useful for hue jitter); LAB is perceptually uniform (distances = perceived
   differences).  Use domain knowledge to choose the right space for your problem.

5. **Normalisation matters**: using the dataset's own statistics (CIFAR-10 mean/std) centres the input
   distribution optimally.  ImageNet statistics introduce a systematic offset.  Per-image normalisation
   removes inter-image brightness information — sometimes helpful, sometimes harmful.

### What's Next

→ **04-05 (Handling Imbalanced Data)** applies the precision-recall framework from 4-01 to datasets
where class frequencies are highly skewed — a problem where augmentation alone is insufficient and
resampling or loss reweighting is required.